In [1]:
!pip install matplotlib

## setup

In [2]:
import os, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from typing import Tuple, Union

In [3]:
F_ALL = "./data/all"
F_ALL_IMAGES = F_ALL + "/images"
F_ALL_MASKS = F_ALL + "/masks"

F_JSRT = "./data/jsrt"
F_JSRT_IMAGES = F_JSRT + "/images"
F_JSRT_MASKS = F_JSRT + "/masks"

F_PADCHEST = "./data/padchest"
F_PADCHEST_IMAGES = F_PADCHEST + "/images"
F_PADCHEST_MASKS = F_PADCHEST + "/masks"

F_SHENZHEN = "./data/shenzhen"
F_SHENZHEN_IMAGES = F_SHENZHEN + "/images"
F_SHENZHEN_MASKS = F_SHENZHEN + "/masks"

F_MONTGOMERY = "./data/montgomery"
F_MONTGOMERY_IMAGES = F_MONTGOMERY + "/images"
F_MONTGOMERY_MASKS = F_MONTGOMERY + "/masks"

F_TEKNOFEST = "./data/teknofest"
F_TEKNOFEST_IMAGES = F_TEKNOFEST + "/images"

CSV_PATH = F_ALL + "/extracted_data.csv"
open(CSV_PATH, "w").close()

In [4]:
class DataExtractor:

    def __init__(self, csv_path, color_left_lung, color_right_lung, color_heart):
        self.csv_path = csv_path
        self.color_left_lung, self.color_right_lung, self.color_heart = color_left_lung, color_right_lung, color_heart

    def get_df(self, n=None):
        df = pd.read_csv(self.csv_path)
        return df.head(n) if n is not None else df

    def extract_names(self, folder):
        names = sorted([
            os.path.splitext(f)[0]
            for f in os.listdir(folder)
            if os.path.isfile(os.path.join(folder, f))
        ])
        df = pd.DataFrame({"name": names})
        df.to_csv(self.csv_path, index=False)

    def extract_genders(self):
        df = self.get_df()

        def assign_gender(name):
            if "-M-" in name:
                return "M"
            elif "-F-" in name:
                return "F"
            else:
                return ""

        df['gender'] = df['name'].apply(assign_gender)
        df.to_csv(self.csv_path, index=False)

    def extract_all_widths_and_ctr(self, masks_folder):
        df = self.get_df()

        for idx, row in df.iterrows():
            base_name = row['name']
            
            img_path = None
            for filename in os.listdir(masks_folder):
                if os.path.splitext(filename)[0] == base_name:
                    img_path = os.path.join(masks_folder, filename)
                    break
            
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            cardiac_width = None
            heart_mask = np.all(img == self.color_heart, axis=2)
            if np.any(heart_mask):
                heart_coords = np.where(heart_mask)
                heart_x_coords = heart_coords[1]
                cardiac_width = np.max(heart_x_coords) - np.min(heart_x_coords) + 1
            
            thoracic_width = 0
            left_lung_mask = np.all(img == self.color_left_lung, axis=2)
            right_lung_mask = np.all(img == self.color_right_lung, axis=2)
            left_lung_coords = np.where(left_lung_mask)
            right_lung_coords = np.where(right_lung_mask)
            
            if len(left_lung_coords[0]) > 0 and len(right_lung_coords[0]) > 0:
                all_y_coords = np.unique(np.concatenate([left_lung_coords[0], right_lung_coords[0]]))
                
                max_distance = 0
                for y in all_y_coords:
                    left_lung_x_at_y = left_lung_coords[1][left_lung_coords[0] == y]
                    right_lung_x_at_y = right_lung_coords[1][right_lung_coords[0] == y]
                    
                    if len(left_lung_x_at_y) > 0 and len(right_lung_x_at_y) > 0:
                        leftmost_left = np.min(left_lung_x_at_y)
                        rightmost_right = np.max(right_lung_x_at_y)
                        distance = rightmost_right - leftmost_left + 1
                        max_distance = max(max_distance, distance)
                
                thoracic_width = max_distance
            
            df.loc[idx, 'thoracic_width'] = thoracic_width if thoracic_width > 0 else None
            df.loc[idx, 'cardiac_width'] = cardiac_width
            
            if cardiac_width is not None and thoracic_width is not None and thoracic_width > 0:
                df.loc[idx, 'ctr'] = f"{cardiac_width / thoracic_width:.3f}"
            else:
                df.loc[idx, 'ctr'] = None
        
        df.to_csv(self.csv_path, index=False)


extractor = DataExtractor(CSV_PATH, (85,85,85), (170,170,170), (255,255,255))

In [5]:
class Debugger:
    
    def __init__(self, csv_path):
        self.csv_path = csv_path


debugger = Debugger(CSV_PATH)

In [6]:
extractor.extract_names(F_ALL_IMAGES)
extractor.get_df()

,name
0,jsrt-JPCLN001
1,jsrt-JPCLN002
2,jsrt-JPCLN003
3,jsrt-JPCLN004
4,jsrt-JPCLN005
...,...
1082,shenzhen-M-1557
1083,shenzhen-M-1559
1084,shenzhen-M-1561
1085,shenzhen-M-1564


## CTR values

In [7]:
extractor.extract_all_widths_and_ctr(F_ALL_MASKS)
extractor.get_df(5)

,name,thoracic_width,cardiac_width,ctr
0,jsrt-JPCLN001,389.0,184.0,0.473
1,jsrt-JPCLN002,362.0,222.0,0.613
2,jsrt-JPCLN003,371.0,184.0,0.496
3,jsrt-JPCLN004,377.0,172.0,0.456
4,jsrt-JPCLN005,373.0,234.0,0.627


## genders

In [8]:
extractor.extract_genders()

## new